# 3D Meteorology UNet — Velocity-Cube Training

Thin notebook wrapper around `utils/training_3d.py`.

The model learns the residual correction produced by `superresolution-nn`'s `cnn` preprocessing:

```
u_reconstructed = input + model(input)
```

Training, checkpointing, early stopping, and evaluation all live in the module; this notebook only
loads the config, previews a cube, runs training, and plots the results.

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path, target: str = 'models', max_up: int = 8) -> Path:
    root = start.resolve()
    for _ in range(max_up):
        if (root / target).exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    raise FileNotFoundError(f"Could not find '{target}' above {start}")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch

from utils.training_3d import (
    build_dataloaders,
    evaluate,
    load_config,
    run_training,
)

plt.rcParams['figure.dpi'] = 130
print('Project root:', ROOT)

In [ ]:
CONFIG_PATH = ROOT / 'configs/sr_unet_meteo_3d_velocity128.yaml'

# Override any config value here, e.g. point at PACE scratch or shorten a smoke run.
OVERRIDES = [
    # 'data.processed_dir=/storage/ice1/.../data/processed/cnn',
    # 'training.epochs=1',
]

cfg = load_config(CONFIG_PATH, OVERRIDES)
print('Processed data:', cfg['data']['processed_dir'])
print('Model:', cfg['model']['architecture'])
print('Epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])
print('Gradient loss:', cfg['loss']['components']['gradient'])

In [ ]:
loaders = build_dataloaders(cfg, splits=('train', 'val'))
for name, loader in loaders.items():
    print(f'{name}: {len(loader.dataset)} cubes, {len(loader)} batches')

inputs, targets = loaders['train'].dataset[0]
dns = inputs + targets
print('Input:', tuple(inputs.shape), '| Target:', tuple(targets.shape))

z = dns.shape[1] // 2
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, field, title in zip(
    axes,
    (inputs[0, z], targets[0, z], dns[0, z], targets[1, z]),
    ('input u (z-slice)', 'target correction u', 'DNS u', 'target correction v'),
):
    im = ax.imshow(field.cpu(), cmap='RdBu_r')
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
result = run_training(cfg, progress=True)
print(result.summary())

In [ ]:
metrics = evaluate(cfg, split='test', save_predictions=False)
metrics

In [ ]:
history = {k: v for k, v in result.history.items() if not k.startswith('_')}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_yscale('log')
axes[0].set_xlabel('epoch')
axes[0].set_title('Total loss')
axes[0].legend()

axes[1].plot(history['train_loss_pixel'], label='train pixel')
axes[1].plot(history['val_loss_pixel'], label='val pixel')
axes[1].plot(history['train_loss_gradient'], label='train gradient')
axes[1].set_yscale('log')
axes[1].set_xlabel('epoch')
axes[1].set_title('Loss components')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
test_loader = build_dataloaders(cfg, splits=('test',), shuffle=False)['test']
test_inputs, test_targets = next(iter(test_loader))
test_inputs = test_inputs.to(result.device)
test_targets = test_targets.to(result.device)

result.model.eval()
with torch.no_grad():
    correction = result.model(test_inputs)

reconstructed = test_inputs + correction
dns = test_inputs + test_targets

z = dns.shape[2] // 2
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, field, title in zip(
    axes,
    (
        dns[0, 0, z],
        reconstructed[0, 0, z],
        (reconstructed - dns)[0, 0, z],
        test_targets[0, 0, z],
    ),
    ('DNS u', 'reconstructed u', 'error', 'target correction u'),
):
    im = ax.imshow(field.cpu(), cmap='RdBu_r')
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()